In [1]:
import torch
torch.cuda.is_available()

True

In [44]:
import torch.nn as nn
class MyConv2D(nn.Module):
    def __init__(self, kernel_size=5, stride=1, padding=0, channel_in=3, channel_out=10):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        self.channel_in = channel_in
        self.channel_out = channel_out
        
        self.kernel = nn.Parameter(torch.randn(channel_out, channel_in * kernel_size * kernel_size))
        
        # Kaiming initialization
        nn.init.kaiming_uniform_(self.kernel, a=0, mode="fan_in", nonlinearity="relu")

        self.unfold = nn.Unfold(kernel_size=kernel_size, stride=stride, padding=padding)
    
    def forward(self, x):
        device = x.device
        # x has the shape (N, n_channels, height, width)
        N, n_channels, h, w = x.shape
        # extract receptive information
        unfolded_tensor = self.unfold(x)
        
        # new shape
        new_h = (h + 2 * self.padding - self.kernel_size) // self.stride + 1
        new_w = (w + 2 * self.padding - self.kernel_size) // self.stride + 1
        
        unfolded_field_tensor = self.kernel @ unfolded_tensor
        
        output = unfolded_field_tensor.reshape(N, -1, new_h, new_w)
        
        
        return output
                

In [45]:
my_conv2d = MyConv2D(kernel_size=2, channel_in=3, channel_out=10).to('cuda')

In [46]:
images = torch.randn(1, 3, 10, 10, device='cuda')
output = my_conv2d(images)

In [47]:
import torch.nn.functional as F
F.relu(output).shape

torch.Size([1, 10, 9, 9])

In [48]:
class MyMaxPooling(nn.Module):
    def __init__(self, kernel_size=2, stride=1, padding=0):
        super().__init__()
        self.kernel_size=kernel_size
        self.stride = stride
        self.padding=padding
        
        self.unfold = nn.Unfold(kernel_size=self.kernel_size, 
                                padding=self.padding, 
                                stride=self.stride)
    def forward(self, x):
        device = x.device
        # x has shape (N, channels, h, w)
        N, channels, h, w = x.shape
        
        # get receptive infomation
        unfolded_tensor = self.unfold(x) # shape: (N, ..., L)
        unfolded_tensor = unfolded_tensor.reshape(N, channels, self.kernel_size * self.kernel_size, -1)
        
        #new height and width
        new_h = (h + 2 * self.padding - self.kernel_size) // self.stride + 1
        new_w = (w + 2 * self.padding - self.kernel_size) // self.stride + 1
        
        pooled_tensor = torch.max(unfolded_tensor, dim=-2).values
        output = pooled_tensor.reshape(N, channels, new_h, new_w)
        
        return output
                
            

In [49]:
max_pooling = MyMaxPooling()
test = torch.randn(1, 3, 3, 3, device='cuda')
max_pooling(test)

tensor([[[[-0.4685,  0.5174],
          [ 1.4616,  1.4616]],

         [[ 1.6649,  1.6649],
          [ 1.9960,  1.9960]],

         [[ 0.9405,  2.3561],
          [ 0.7065,  1.8113]]]], device='cuda:0')

In [50]:
images = torch.randn(1, 3, 10, 10, device='cuda')
output = my_conv2d(images)

In [51]:
output = F.relu(output)
output.shape

torch.Size([1, 10, 9, 9])

In [52]:
output = max_pooling(output)
output.shape

torch.Size([1, 10, 8, 8])

In [53]:
import torch.nn.functional as F
class MyOwnCNN(nn.Module):
    def __init__(self, num_classes=3, channel_in=1):
        super().__init__()
        # C1
        self.c1 = MyConv2D(kernel_size=5, 
                           padding=2, 
                           stride=1, 
                           channel_in=channel_in, 
                           channel_out=16)
        
        # C4 
        self.c4 = MyConv2D(kernel_size=5,
                           padding=2,
                           stride=1,
                           channel_in=16,
                           channel_out=32)
        
        # C7 
        self.c7 = MyConv2D(kernel_size=5,
                           padding=2,
                           stride=1,
                           channel_in=32,
                           channel_out=64)
        
        # MaxPooling
        self.max = MyMaxPooling(stride=2)
        
        #flatten before forward to fc
        self.flatten = nn.Flatten()
        
        with torch.no_grad():
            x = torch.zeros(1, 1, 28, 28)     # dummy image
            x = self.max(F.relu(self.c1(x)))
            x = self.max(F.relu(self.c4(x)))
            x = self.max(F.relu(self.c7(x)))
            flat_dim = x.numel()             # number of features
        
        # two fcs layers
        self.fc1 = nn.Linear(flat_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = F.relu(self.c1(x))
        x = self.max(x)
        x = F.relu(self.c4(x))
        x = self.max(x)
        x = F.relu(self.c7(x))
        x = self.max(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        
        return logits

In [54]:
my_cnn = MyOwnCNN().to('cuda')

In [55]:
x = torch.randn(1, 1, 28, 28, device='cuda')

my_cnn(x)

tensor([[0.1446, 0.1402, 0.0070]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [56]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Define transformations
# Convert images from PIL Image format (or numpy array) to PyTorch tensors
# and normalize them. MNIST images are grayscale, so we use single mean/std dev.
transform = transforms.Compose([
    transforms.ToTensor(),
    # Normalize with mean 0.5 and std dev 0.5 for a single channel
    transforms.Normalize((0.5,), (0.5,)) 
])

# 2. Load the training dataset
# 'root' specifies where to save the data
# 'train=True' gets the training set (60,000 images)
# 'download=True' downloads the data if not already present
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

# 3. Load the test dataset
# 'train=False' gets the test set (10,000 images)
test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# 4. Create DataLoaders for efficient iteration and batching
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,          # Shuffle training data
    num_workers=2          # Use multiple subprocesses for data loading (optional, common for performance)
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,         # No need to shuffle test data
    num_workers=2
)

# 5. Example of iterating through the DataLoader
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Get one batch of data to verify
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"\nShape of an image batch: {images.shape}")  # e.g., torch.Size([64, 1, 28, 28])
print(f"Shape of a label batch: {labels.shape}")    # e.g., torch.Size([64])


Number of training batches: 938
Number of test batches: 157

Shape of an image batch: torch.Size([64, 1, 28, 28])
Shape of a label batch: torch.Size([64])


In [57]:
digit_classifier = MyOwnCNN(num_classes=10).to('cuda')

In [58]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
NUM_EPOCHS = 5
loss_fn = nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
optimizer = optim.Adam(digit_classifier.parameters(), lr=.001)
for epoch in range(NUM_EPOCHS):
    digit_classifier.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in tqdm(enumerate(train_loader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", total=len(train_loader)):
        images = images.to(device)
        labels = labels.to(device)  
        
        optimizer.zero_grad()
        
        outputs = digit_classifier(images)
        
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f'---- End of Epoch {epoch+1}, Average Loss: {running_loss/len(train_loader):.4f} ----')

Epoch 1/5: 100%|██████████| 938/938 [00:13<00:00, 67.67it/s]

---- End of Epoch 1, Average Loss: 0.1491 ----



Epoch 2/5: 100%|██████████| 938/938 [00:15<00:00, 59.86it/s]

---- End of Epoch 2, Average Loss: 0.0435 ----



Epoch 3/5: 100%|██████████| 938/938 [00:14<00:00, 63.45it/s] 

---- End of Epoch 3, Average Loss: 0.0318 ----



Epoch 4/5: 100%|██████████| 938/938 [00:15<00:00, 60.02it/s]

---- End of Epoch 4, Average Loss: 0.0244 ----



Epoch 5/5: 100%|██████████| 938/938 [00:12<00:00, 78.02it/s] 

---- End of Epoch 5, Average Loss: 0.0211 ----


**Note**: Without `Kaiming` initialization, training doesn't work

In [61]:
a = torch.tensor([[1, 2, 3], [4, 2, 1]])
torch.max(a, dim=-1).indices.sum()

tensor(2)

In [63]:
acc = 0
for images, labels in tqdm(test_loader):
    labels = labels.to('cuda')
    images = images.to('cuda')
    
    logits = digit_classifier(images)
    predicted_values = torch.max(logits, dim=-1).indices
    acc += (predicted_values == labels).sum().item()

print(acc / len(test_dataset))
    

100%|██████████| 157/157 [00:02<00:00, 72.48it/s]

0.9904
